# Activity 1 — Are these groups real?

**MATH4SDG / UDSM · Student investigation · 10–15 minutes plus discussion**

You have measurements of **90 objects**, with two numerical measurements per object. The file contains entirely **synthetic classroom data**, not real information from an industrial partner. The mechanism used to construct the numbers will be discussed with the facilitator later.

Your task is to explore the data, propose two groups, and test whether their membership survives fresh independent measurements of the same objects. **Write down your interpretation before looking at the second dataset.**

## 1. Inspect the first measurements

Keep this notebook inside `notebooks/` and its CSV files in `data/exercises/`, as supplied in the repository.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score

first = pd.read_csv("../data/exercises/clustering_first.csv")
assert first["sample_id"].is_unique
print("Objects:", len(first))
print("Missing entries:", first.isna().sum().to_dict())
first.head()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
ax.scatter(first["measurement_1"], first["measurement_2"], color="gray", s=38, alpha=0.8)
ax.set(xlabel="Measurement 1", ylabel="Measurement 2", title="First measurements — no groups supplied")
ax.set_aspect("equal", adjustable="box")
plt.show()

**Before clustering:** Do these measurements suggest distinct populations? Why, or why not?

## 2. Ask K-means to find two candidate groups

We choose `n_clusters=2` ourselves. That makes the algorithm produce two groups, regardless of whether two natural populations exist.

In [ ]:
columns = ["measurement_1", "measurement_2"]
X = first[columns].to_numpy()
labels = KMeans(n_clusters=2, n_init=20, random_state=42).fit_predict(X)
first["candidate_group"] = labels
print("Sizes:", first["candidate_group"].value_counts().sort_index().to_dict())

fig, ax = plt.subplots(figsize=(6, 5))
for g in (0, 1):
    part = first.loc[first["candidate_group"] == g]
    ax.scatter(part["measurement_1"], part["measurement_2"],
               s=38, alpha=0.85, label=f"Candidate group {g + 1}")
ax.set(xlabel="Measurement 1", ylabel="Measurement 2", title="Two discovered groups")
ax.set_aspect("equal", adjustable="box")
ax.legend()
plt.show()

In [ ]:
original_score = silhouette_score(X, labels)
print(f"Silhouette score in the discovery data: {original_score:.3f}")

### Stop and record your hypothesis

Discuss with a partner **before running the next cell**.

- Do the two colored groups appear convincing?
- What might they represent if these were biological or industrial measurements?
- What observation would convince you that these are stable groups?

**Our interpretation:** *(write your answer here)*

**Our proposed test:** *(write your answer here)*

## 3. Independent fresh measurements

You now receive **new, independent measurements of the same 90 object IDs**. Keep the original group labels fixed: do not run K-means again before checking them.

New measurements may not always be independent in real applications. Here they are deliberately constructed to provide a simple reproducibility test.

In [ ]:
fresh = pd.read_csv("../data/exercises/clustering_fresh.csv")
comparison = first[["sample_id", "candidate_group"]].merge(
    fresh, on="sample_id", validate="one_to_one"
)
assert len(comparison) == len(first) == len(fresh)

fig, ax = plt.subplots(figsize=(6, 5))
for g in (0, 1):
    part = comparison.loc[comparison["candidate_group"] == g]
    ax.scatter(part["measurement_1"], part["measurement_2"],
               s=38, alpha=0.85, label=f"Original group {g + 1}")
ax.set(xlabel="Fresh measurement 1", ylabel="Fresh measurement 2",
       title="Fresh measurements — original labels retained")
ax.set_aspect("equal", adjustable="box")
ax.legend()
plt.show()

In [ ]:
Y = comparison[columns].to_numpy()
fresh_score = silhouette_score(Y, comparison["candidate_group"].to_numpy())
print(f"Discovery data: {original_score:.3f}")
print(f"Fresh data with the SAME labels: {fresh_score:.3f}")

**Discuss:** Does the discovered grouping persist? Why would finding *another* partition in the fresh data fail to establish that the original groups were genuine?

## 4. Optional: compare two independently constructed partitions

Run K-means again on the new measurements, but **compare its labels with the old ones for the same objects**. The adjusted Rand index accounts for arbitrary exchanges of group names.

In [ ]:
new_labels = KMeans(n_clusters=2, n_init=20, random_state=42).fit_predict(Y)
print("Adjusted Rand index:", round(adjusted_rand_score(labels, new_labels), 3))

## Bring to the group discussion

1. What does K-means establish, and what does it not establish?
2. What role did the choice of `n_clusters=2` play?
3. What further evidence would justify interpreting these as scientifically meaningful categories?

**The facilitator will discuss how the data were constructed after the investigation.**